# Callable stepped-coupon bond

An amortizing USD rates note with fixed and capped/floored floating coupons, inclusive issuer-call windows, and OAS risk. Contract data is a native Python dictionary; discounting is selected by the pricing context, not embedded in the bond details.

The market is the shipped Hull-White snapshot dated 2026-02-24. Both native engines calibrate a short-rate volatility schedule to ATM caplets sampled from its constructed SOFR Black-volatility surface. No fixed sigma is used in the main valuation.

In [1]:
import json
from pathlib import Path

import quantsupport as qs

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "examples" / "hullwhite" / "data" / "quotes.json").is_file()
)
DATA = ROOT / "examples" / "hullwhite" / "data"
with (DATA / "quotes.json").open(encoding="utf-8") as stream:
    quote_data = json.load(stream)
with (DATA / "curve_specs.json").open(encoding="utf-8") as stream:
    curve_data = json.load(stream)
with (DATA / "vol_specs.json").open(encoding="utf-8") as stream:
    volatility_data = json.load(stream)

quotes = qs.QuoteStore.from_dict(quote_data)
reference_date = quotes.reference_date
curves = [
    qs.CurveConfiguration.from_dict(spec)
    for spec in curve_data["curve_specs"]
]
volatility_surfaces = [
    qs.VolatilitySurfaceConfiguration.from_dict(spec)
    for spec in volatility_data["volatility_surfaces"]
]
discounting = qs.DiscountingConfig.from_dict({"currency": "USD", "index": "SOFR"})
assert str(reference_date) == quote_data["reference_date"]
print("Market reference date:", reference_date)
print("Caplet surface quotes:", len(volatility_data["volatility_surfaces"][0]["quotes"]))

Market reference date: 2026-02-24
Caplet surface quotes: 27


## Contract terms

The first coupon is fixed; the floating definition repeats for all later periods. Outstanding principal steps from 100% to 90% to 75%, with the last percentage carried through maturity. Principal reductions pay at period ends. Each call entry is an inclusive start/end window with a strike quoted per 100 of outstanding principal; the windows do not overlap.

In [2]:
bond_data = {
    "identifier": "USD-RATES-STEP-CALL-2032",
    "start_date": str(reference_date),
    "maturity_date": str(reference_date + "6Y"),
    "notional": 1_000_000.0,
    "units": 100.0,
    "currency": "USD",
    "side": "LongReceive",
    "payment_frequency": "Semiannual",
    "coupon_day_counter": "Actual365",
    "coupons": [
        {"Fixed": {"rate": 0.045}},
        {"Floating": {"index": "SOFR", "spread": 0.006, "floor": 0.0, "cap": 0.06}},
    ],
    "notional_percentages": [1.0, 0.90, 0.75],
    "call_schedule": [
        {"start_date": str(reference_date + "3Y"),
         "end_date": str(reference_date + "3Y" + "3M"), "strike": 101.0},
        {"start_date": str(reference_date + "4Y"),
         "end_date": str(reference_date + "4Y" + "3M"), "strike": 100.0},
    ],
}
bond = qs.SteppedCouponBond.from_dict(bond_data)
bond

SteppedCouponBond(identifier='USD-RATES-STEP-CALL-2032', start_date='2026-02-24', maturity_date='2032-02-24', notional=1000000, coupons=2, notionals=3, calls=2)

In [3]:
calibration = {
    "source": {"Surface": {"market_index": "SOFR"}},
    "calibration_basket": {"strike": "Atm"},
}
model = {
    "market_index": "SOFR",
    "model": {
        "HullWhite": {
            "alpha": 0.05,
            "parameter_source": {"Calibrated": calibration},
        }
    },
}
time_grid = {
    "steps_per_year": 96, "max_steps": 4096, "day_counter": "Actual365",
}
tree_configuration = qs.PricerConfiguration.from_dict({
    "instrument_type": "SteppedCouponBond",
    "pricer": {"TreeBased": {"time_grid": time_grid}},
    "models": [model],
})
fourier_configuration = qs.PricerConfiguration.from_dict({
    "instrument_type": "SteppedCouponBond",
    "pricer": {
        "BackwardFourierEvolution": {
            "time_grid": time_grid,
            "fourier_grid": {"points": 512, "standard_deviations": 10.0},
        }
    },
    "models": [model],
})

In [4]:
def price(configuration):
    with qs.PricingContext(
        quotes=quotes, curves=curves, discounting=discounting,
        volatility_surfaces=volatility_surfaces,
        pricer_configurations=[configuration],
    ) as context:
        result = context.evaluate(
            bond,
            [qs.Request.Value, qs.Request.OAS, qs.Request.Sensitivities],
            bond_anchor=qs.BondPricingAnchor.OAS(0.0075),
        )
        surface_nodes = context.volatility_surface(qs.MarketIndex.SOFR).pillars()
        return result, surface_nodes

tree, surface_nodes = price(tree_configuration)
fourier, _ = price(fourier_configuration)
print(f"Tree PV:             {tree.price:,.8f}")
print(f"Backward Fourier PV: {fourier.price:,.8f}")
print(f"Absolute difference: {abs(tree.price - fourier.price):.6f}")
print("Constructed surface pillars:", len(surface_nodes))
assert len(surface_nodes) > 0
tree.sensitivities

QuantSupportError: Invalid value error: Callable-bond lattices currently require fixed Hull-White parameters; calibrated parameters must be resolved during PricingContext initialization

Yield-to-maturity and modified duration are intentionally not exposed for this product. It is anchored and risked by clean/dirty price or OAS. The OAS derivative is returned alongside curve and calibrated option-quote risk.

## Inline scripted cashflows on the calibrated market

The script below reproduces the fixed/floating coupons and amortization of the **non-callable underlying**, at zero OAS. Each event fixes its coupon at the accrual start and schedules coupon plus principal at the end. It uses the same market surface and calibrated SOFR parameter source through the Python module. It is not an optimal issuer-exercise algorithm, so its value should not be equated with the callable, positive-OAS prices above.

In [ ]:
script_data = {
    "identifier": "USD-RATES-STEP-CASHFLOWS-2032",
    "currency": "USD",
    "result_variable": "bond",
    "events": [],
}
payment_dates = [reference_date + f"{6 * period}M" for period in range(13)]
for period, (start, end) in enumerate(zip(payment_dates, payment_dates[1:])):
    coupon = bond_data["coupons"][min(period, len(bond_data["coupons"]) - 1)]
    percentages = bond_data["notional_percentages"]
    outstanding = percentages[min(period, len(percentages) - 1)]
    next_outstanding = percentages[min(period + 1, len(percentages) - 1)] if period < 11 else 0.0
    principal = bond_data["notional"] * (outstanding - next_outstanding)
    if "Fixed" in coupon:
        rate = str(coupon["Fixed"]["rate"])
        day_counter = bond_data["coupon_day_counter"]
    else:
        floating = coupon["Floating"]
        rate = (
            f'max(min(RateIndex("{floating["index"]}", "{start}", "{end}") '
            f'+ {floating["spread"]}, {floating["cap"]}), {floating["floor"]})'
        )
        day_counter = "Actual360"
    initialization = "bond = 0; " if period == 0 else ""
    script_data["events"].append({
        "date": str(start),
        "code": (
            f'{initialization}accrual = cvg("{start}", "{end}", "{day_counter}"); '
            f'rate = {rate}; '
            f'bond pays {bond_data["notional"] * outstanding} * rate * accrual '
            f'+ {principal} on "{end}";'
        ),
    })
script = qs.ScriptedProduct.from_dict(script_data)
with qs.PricingContext(
    quotes=quotes, curves=curves, discounting=discounting,
    volatility_surfaces=volatility_surfaces,
    pricer_configurations=[tree_configuration],
) as context:
    scripted = context.evaluate_script(script, n_paths=4096, seed=42)
print(f"Calibrated scripted underlying PV (zero OAS): {scripted['bond']:,.8f}")

## Isolated deterministic representation check

This validation intentionally removes issuer calls and sets sigma to zero in a **separate** context. It compares the same underlying coupon/amortization contract, at zero OAS, in native and scripted representations. It is not the model specification used for the calibrated valuations above and does not validate stochastic issuer-call exercise. The tolerance is 0.05% of value to allow for tree-grid discretization.

In [ ]:
import math

validation_bond = qs.SteppedCouponBond.from_dict({
    **bond_data, "identifier": "USD-RATES-STEP-REPRESENTATION-CHECK",
    "call_schedule": [],
})
validation_configuration_data = tree_configuration.to_dict()
validation_configuration_data["models"][0]["model"]["HullWhite"]["parameter_source"] = {
    "Fixed": {"sigma": 0.0},
}
validation_configuration = qs.PricerConfiguration.from_dict(validation_configuration_data)
with qs.PricingContext(
    quotes=quotes, curves=curves, discounting=discounting,
    pricer_configurations=[validation_configuration],
) as validation_context:
    native_validation = validation_context.evaluate(
        validation_bond, [qs.Request.Value], bond_anchor=qs.BondPricingAnchor.z_spread(0.0),
    )
    scripted_validation = validation_context.evaluate_script(script, n_paths=1, seed=42)
assert math.isfinite(native_validation.price) and math.isfinite(scripted_validation["bond"])
assert math.isclose(native_validation.price, scripted_validation["bond"], rel_tol=5e-4, abs_tol=1e-8)
print(f"Deterministic native underlying:   {native_validation.price:,.8f}")
print(f"Deterministic scripted underlying: {scripted_validation['bond']:,.8f}")
print(f"Absolute difference: {abs(native_validation.price - scripted_validation['bond']):.6f}")